In [79]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

METHOD = "hmm_pomegranate"


# 1. Read train data

In [80]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/"


In [81]:
from data_reading_methods import get_available_alleles, read_data
import numpy as np
ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

total_seen_peptides_by_models = \
set(np.concatenate([per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9], per_allele_per_kfold_per_length_binders_test[ALLELES[0]][0][9]]))

ALLELES

['HLA-A*02:01']

# Read External benchmark

In [83]:

TARGET_ALLELE = ALLELES[0]
TARGET_LENGTH = 9

In [128]:
import pandas as pd
from mhcgnomes import parse
pearson_df = pd.read_csv("D:\\Google Drive\\Work\\ITMO\\data\\Binding data\\Pearson\\Pearson_dataset_sup2.csv")
pearson_df['allele'] = pearson_df['Allele'].apply(lambda x:":".join(x.rsplit("-", 1))).apply(lambda x: parse(x).to_string())
pearson_df = pearson_df[pearson_df['allele'] == TARGET_ALLELE]
pearson_df = pearson_df[pearson_df['Length'] == TARGET_LENGTH]
pearson_df['label'] = 0
pearson_df.loc[pearson_df['Binding Affinity'] < 500, 'label'] = 1
pearson_df = pearson_df.rename(columns={"Peptide Sequence": "peptide", "Length": "length"})
pearson_df.head()

,peptide,length,Binding Affinity,Allele,Subject ID,Chromosome,Gene Symbol,Gene ID,Transcript ID,Protein ID,Gene Expression,Position in Protein,allele,label
21,LLPPPPCPA,9,531.0,HLA-A*02-01,3,11,FUCA2,ENSG00000001036,ENST00000002165,ENSP00000002165,26.83,19,HLA-A*02:01,0
22,LLPPPPCPA,9,531.0,HLA-A*02-01,4,11,FUCA2,ENSG00000001036,ENST00000002165,ENSP00000002165,17.18,19,HLA-A*02:01,0
23,LLPPPPCPA,9,531.0,HLA-A*02-01,6,11,FUCA2,ENSG00000001036,ENST00000002165,ENSP00000002165,20.84,19,HLA-A*02:01,0
24,LLPPPPCPA,9,531.0,HLA-A*02-01,7,11,FUCA2,ENSG00000001036,ENST00000002165,ENSP00000002165,9.31,19,HLA-A*02:01,0
25,LLPPPPCPA,9,531.0,HLA-A*02-01,8,11,FUCA2,ENSG00000001036,ENST00000002165,ENSP00000002165,12.53,19,HLA-A*02:01,0


In [129]:
pearson_df['label'].value_counts()

1    5662
0     187
Name: label, dtype: int64

In [130]:
len(pearson_df['peptide'].unique())

1845

In [131]:
len(pearson_df[pearson_df['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique())

1845

## Can't use pearson dataset, it's already in IEDB.
# Try Genentech test data

In [138]:
gne_df =  pd.read_csv("D:\Google Drive\Work\ITMO\data\genentech\gne_benchmark_class-I.csv")
gne_df['allele'] = gne_df['allele'].apply(lambda x: parse(x).to_string())
gne_df = gne_df[gne_df['allele'] == TARGET_ALLELE]
gne_df = gne_df[gne_df['split'] == 'test']
gne_df['length'] = gne_df['peptide'].str.len() 
gne_df = gne_df[gne_df['length'] == TARGET_LENGTH]

d:\tools\python\anaconda\envs\pomegranate-env\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (6,7,8,9,14,21,22,23,24) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [139]:
genentech_binders =gne_df[gne_df['el'] == 1]
genentech_non_binders = gne_df[gne_df['el'] == 0]
len(genentech_binders['peptide'].unique()), len(genentech_non_binders['peptide'].unique())

(367, 9596)

In [140]:
len(genentech_binders[genentech_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique()), \
len(genentech_non_binders[genentech_non_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique())

(365, 18)

In [141]:
validation_binders = genentech_binders[~genentech_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique()
validation_non_binders  = genentech_non_binders[~genentech_non_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique()
len(validation_binders), len(validation_non_binders)

(2, 9578)

## Can't use genentech dataset, it's already in IEDB.

## NetMHCpan4.0 dataset

In [162]:
ba_df_train = pd.read_csv("D:\\Google Drive\\Work\\ITMO\\data\\Binding data\\NetMHCpan-4.0\\f000_ba.txt", sep=' ', header=None)

ba_df_test = pd.read_csv("D:\\Google Drive\\Work\\ITMO\\data\\Binding data\\NetMHCpan-4.0\\c000_ba.txt", sep=' ', header=None)

ba_df_total = pd.concat([ba_df_train, ba_df_test])
ba_df_total = ba_df_total.rename(columns={0:"peptide", 1:"score", 2:"allele", 3:"affinity"})
ba_df_total = ba_df_total[ba_df_total['allele'].apply(lambda x: parse(x).to_string()) == TARGET_ALLELE]
ba_df_total['length'] = ba_df_total['peptide'].str.len()
ba_df_total = ba_df_total[ba_df_total['length'] == TARGET_LENGTH]
ba_df_total['label'] = 0
ba_df_total.loc[ba_df_total['score'] > 0.42, 'label'] = 1

ba_df_total =ba_df_total[['peptide', 'allele','label']]


el_df_train = pd.read_csv("D:\\Google Drive\\Work\\ITMO\\data\\Binding data\\NetMHCpan-4.0\\f000_el.txt", sep='\t', header=None)
el_df_test = pd.read_csv("D:\\Google Drive\\Work\\ITMO\\data\\Binding data\\NetMHCpan-4.0\\c000_el.txt", sep='\t', header=None)
el_df_total = pd.concat([el_df_train, el_df_test])
el_df_total = el_df_total.rename(columns={0:"peptide", 1:"label", 2:"allele"})
el_df_total = el_df_total[el_df_total['allele'].apply(lambda x: parse(x).to_string()) == TARGET_ALLELE]
el_df_total['length'] = el_df_total['peptide'].str.len()
el_df_total = el_df_total[el_df_total['length'] == TARGET_LENGTH]

el_df_total = el_df_total[['peptide', 'allele','label']]


netmhc_total = pd.concat([ba_df_total, el_df_total])


netmhc_total.head()

,peptide,allele,label
2909,AADKAAAAY,HLA-A02:01,0
2910,AAGAAVKGV,HLA-A02:01,0
2911,AAGIGILTV,HLA-A02:01,1
2912,AAHARFVAA,HLA-A02:01,0
2913,AAICTHLEV,HLA-A02:01,0


In [163]:
netmhc_binders = netmhc_total[netmhc_total['label'] == 1]
netmhc_non_binders =  netmhc_total[netmhc_total['label'] == 0]
len(netmhc_binders['peptide'].unique()), len(netmhc_non_binders['peptide'].unique())

(8439, 57979)

In [164]:
len(netmhc_binders[netmhc_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique()), \
len(netmhc_non_binders[netmhc_non_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique())

(8053, 1979)

In [165]:
validation_binders = netmhc_binders[~netmhc_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique()
validation_non_binders  = netmhc_non_binders[~netmhc_non_binders['peptide'].isin(total_seen_peptides_by_models)]['peptide'].unique()
len(validation_binders), len(validation_non_binders)

(386, 56000)

In [166]:
validation_binders

array(['AIIRILQQA', 'ALDAYHASL', 'ALDWIGERL', 'ALFDGAKPL', 'ALPEDYDRV',
       'ALSSSLGNV', 'AMAAQLQAV', 'AMRWGHLPL', 'AMVDGVSVL', 'AMYRDMATI',
       'AQLQAVPGA', 'ASAKAAAAV', 'AVNGVMWTV', 'FLGFLATAG', 'FLLDGGAPF',
       'FLLRHLSSV', 'FLLVAHYAI', 'FLPDKAIDL', 'FLSHYFTLV', 'FLVSQLFTF',
       'FLYNRPLNS', 'FMYSTVATI', 'FQAQNIAGL', 'GLLEQAAAV', 'GLLSSKFKA',
       'GLVESVAGS', 'GLVFHSQPI', 'GMLFGAAGV', 'GMVGAEFSL', 'HVDIRTLED',
       'IFLLVLLDY', 'ILFCFLAAV', 'KLTPGFHGL', 'LLAAGADEV', 'LLDEPTNNL',
       'LLLAILGPL', 'LLMPKTASS', 'LLQEMVEYV', 'LLYMTSATI', 'LMKMGAPEV',
       'LTAAVLMLA', 'LTMKTNMLV', 'MLSATGLSL', 'NLTEEMAAL', 'QILEENVEV',
       'RLFFYRKSV', 'RLQDGEPYL', 'RLSKRSYLI', 'RLWHYPCTA', 'RQMRASAPL',
       'SIFFDYMAI', 'SLAYNKFYI', 'SLQDNAVTV', 'SLYNTVATI', 'TLKEDTMEV',
       'TLKEGTMEV', 'TLPDVKSFV', 'VILWFSFGA', 'VLAAGADEV', 'VLFDDLCEA',
       'VLFTVLAIV', 'VLLDSITRL', 'WLPSGGTLV', 'WLSQTTLSV', 'WMYRQQNPI',
       'YLDWHAGHA', 'YLFFYRKSV', 'YLFYDFLLV', 'YLGDEILEV', 'YLIK